# Pipeline Batch — Indicador Criança Alfabetizada

Este notebook simula a execução periódica da ingestão batch do projeto Tech Challenge — Fase 2.

## Objetivos

- Extrair dados históricos da Base dos Dados;
- Carregar as entidades na camada Bronze do BigQuery;
- Preservar os dados sem transformações significativas;
- Validar os volumes entre origem e destino;
- Registrar duração, status e identificador da execução;
- Tornar o processo reexecutável.

## Arquitetura

Base dos Dados → Ingestão Batch → Bronze → Validação → Silver → Gold

> A execução é acionada manualmente devido às limitações do BigQuery Sandbox, mas o código foi estruturado para processamento periódico.

1. AUTENTICAÇÃO E CONFIGURAÇÃO DO PIPELINE

In [2]:


from google.colab import auth
from google.cloud import bigquery

from datetime import datetime, timezone
import time
import uuid
import pandas as pd


# Abre a autenticação com a conta Google
auth.authenticate_user()


# ------------------------------------------------------------
# Configurações do projeto
# ------------------------------------------------------------

PROJECT_ID = "macro-coil-475920-k5"
LOCATION = "US"

SOURCE_PROJECT = "basedosdados"
SOURCE_DATASET = "br_inep_avaliacao_alfabetizacao"

BRONZE_DATASET = "bronze"


# ------------------------------------------------------------
# Identificação desta execução
# ------------------------------------------------------------

execution_id = str(uuid.uuid4())

started_at = datetime.now(timezone.utc)


# ------------------------------------------------------------
# Cliente do BigQuery
# ------------------------------------------------------------

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)


print("Autenticação concluída.")
print(f"Projeto: {PROJECT_ID}")
print(f"Localização: {LOCATION}")
print(f"ID da execução: {execution_id}")
print(f"Início UTC: {started_at.isoformat()}")

Autenticação concluída.
Projeto: macro-coil-475920-k5
Localização: US
ID da execução: e006db26-a760-4bb5-8ba1-65b4a2b44c17
Início UTC: 2026-07-12T13:34:30.344736+00:00


2. MAPA DAS ENTIDADES E DIAGNÓSTICO INICIAL

In [3]:


from google.api_core.exceptions import NotFound, Forbidden


# Entidades exigidas no projeto
TABLES = [
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf",
    "municipio",
    "uf",
]


diagnostico = []


for table_name in TABLES:

    source_table_id = (
        f"{SOURCE_PROJECT}.{SOURCE_DATASET}.{table_name}"
    )

    destination_table_id = (
        f"{PROJECT_ID}.{BRONZE_DATASET}.{table_name}"
    )

    # --------------------------------------------------------
    # Consulta metadados da tabela de origem
    # --------------------------------------------------------

    try:
        source_table = client.get_table(source_table_id)

        source_exists = True
        source_rows = source_table.num_rows
        source_bytes = source_table.num_bytes

    except (NotFound, Forbidden) as error:
        source_exists = False
        source_rows = None
        source_bytes = None

        print(
            f"Problema ao acessar a origem {source_table_id}: "
            f"{error}"
        )

    # --------------------------------------------------------
    # Consulta metadados da tabela Bronze
    # --------------------------------------------------------

    try:
        destination_table = client.get_table(
            destination_table_id
        )

        destination_exists = True
        destination_rows = destination_table.num_rows
        destination_bytes = destination_table.num_bytes

    except NotFound:
        destination_exists = False
        destination_rows = None
        destination_bytes = None

    # --------------------------------------------------------
    # Registra o diagnóstico
    # --------------------------------------------------------

    diagnostico.append(
        {
            "tabela": table_name,
            "origem_existe": source_exists,
            "linhas_origem": source_rows,
            "mb_origem": (
                round(source_bytes / 1024**2, 2)
                if source_bytes is not None
                else None
            ),
            "bronze_existe": destination_exists,
            "linhas_bronze": destination_rows,
            "mb_bronze": (
                round(destination_bytes / 1024**2, 2)
                if destination_bytes is not None
                else None
            ),
            "contagem_atual_confere": (
                source_rows == destination_rows
                if (
                    source_rows is not None
                    and destination_rows is not None
                )
                else False
            ),
        }
    )


diagnostico_df = pd.DataFrame(diagnostico)

display(diagnostico_df)

,tabela,origem_existe,linhas_origem,mb_origem,bronze_existe,linhas_bronze,mb_bronze,contagem_atual_confere
0,alunos,True,3867999,256.10,True,3867999,256.10,True
1,dicionario,True,27,0.00,True,27,0.00,True
2,meta_alfabetizacao_brasil,True,3,0.00,True,3,0.00,True
3,meta_alfabetizacao_municipio,True,10704,1.10,True,10704,1.10,True
4,meta_alfabetizacao_uf,True,81,0.01,True,81,0.01,True
5,municipio,True,23995,1.75,True,23995,1.75,True
6,uf,True,145,0.01,True,145,0.01,True


3. EXECUÇÃO DA CARGA BATCH PARA A CAMADA BRONZE

In [4]:

batch_logs = []

print("=" * 70)
print("INÍCIO DA EXECUÇÃO BATCH")
print(f"Execution ID: {execution_id}")
print("=" * 70)


for table_name in TABLES:

    source_table_id = (
        f"{SOURCE_PROJECT}.{SOURCE_DATASET}.{table_name}"
    )

    destination_table_id = (
        f"{PROJECT_ID}.{BRONZE_DATASET}.{table_name}"
    )

    table_started_at = datetime.now(timezone.utc)
    timer_start = time.perf_counter()

    print(f"\nProcessando: {table_name}")

    sql = f"""
    CREATE OR REPLACE TABLE
      `{destination_table_id}`

    AS

    SELECT
      *

    FROM
      `{source_table_id}`
    """

    try:
        # Executa a cópia em lote
        query_job = client.query(
            sql,
            location=LOCATION,
            job_config=bigquery.QueryJobConfig(
                labels={
                    "pipeline": "alfabetizacao",
                    "tipo": "batch",
                    "camada": "bronze",
                }
            ),
        )

        query_job.result()

        # Consulta os metadados depois da carga
        source_table = client.get_table(source_table_id)
        destination_table = client.get_table(
            destination_table_id
        )

        duration_seconds = round(
            time.perf_counter() - timer_start,
            2
        )

        source_rows = source_table.num_rows
        destination_rows = destination_table.num_rows

        counts_match = (
            source_rows == destination_rows
        )

        status = (
            "SUCESSO"
            if counts_match
            else "DIVERGENCIA_DE_CONTAGEM"
        )

        print(
            f"  Status: {status} | "
            f"Origem: {source_rows:,} | "
            f"Bronze: {destination_rows:,} | "
            f"Duração: {duration_seconds}s"
        )

        batch_logs.append(
            {
                "execution_id": execution_id,
                "tabela": table_name,
                "inicio_utc": table_started_at.isoformat(),
                "fim_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "status": status,
                "linhas_origem": source_rows,
                "linhas_bronze": destination_rows,
                "contagem_confere": counts_match,
                "duracao_segundos": duration_seconds,
                "bytes_processados": (
                    query_job.total_bytes_processed
                    or 0
                ),
                "mensagem_erro": None,
            }
        )

    except Exception as error:

        duration_seconds = round(
            time.perf_counter() - timer_start,
            2
        )

        print(f"  Status: ERRO")
        print(f"  Motivo: {error}")

        batch_logs.append(
            {
                "execution_id": execution_id,
                "tabela": table_name,
                "inicio_utc": table_started_at.isoformat(),
                "fim_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "status": "ERRO",
                "linhas_origem": None,
                "linhas_bronze": None,
                "contagem_confere": False,
                "duracao_segundos": duration_seconds,
                "bytes_processados": None,
                "mensagem_erro": str(error),
            }
        )


batch_logs_df = pd.DataFrame(batch_logs)

print("\n" + "=" * 70)
print("RESUMO DA EXECUÇÃO")
print("=" * 70)

display(batch_logs_df)

INÍCIO DA EXECUÇÃO BATCH
Execution ID: e006db26-a760-4bb5-8ba1-65b4a2b44c17

Processando: alunos
  Status: SUCESSO | Origem: 3,867,999 | Bronze: 3,867,999 | Duração: 5.51s

Processando: dicionario
  Status: SUCESSO | Origem: 27 | Bronze: 27 | Duração: 3.77s

Processando: meta_alfabetizacao_brasil
  Status: SUCESSO | Origem: 3 | Bronze: 3 | Duração: 3.63s

Processando: meta_alfabetizacao_municipio
  Status: SUCESSO | Origem: 10,704 | Bronze: 10,704 | Duração: 2.88s

Processando: meta_alfabetizacao_uf
  Status: SUCESSO | Origem: 81 | Bronze: 81 | Duração: 3.93s

Processando: municipio
  Status: SUCESSO | Origem: 23,995 | Bronze: 23,995 | Duração: 2.96s

Processando: uf
  Status: SUCESSO | Origem: 145 | Bronze: 145 | Duração: 3.27s

RESUMO DA EXECUÇÃO


,execution_id,tabela,inicio_utc,fim_utc,status,linhas_origem,linhas_bronze,contagem_confere,duracao_segundos,bytes_processados,mensagem_erro
0,e006db26-a760-4bb5-8ba1-65b4a2b44c17,alunos,2026-07-12T13:39:57.241113+00:00,2026-07-12T13:40:02.747436+00:00,SUCESSO,3867999,3867999,True,5.51,268545240,None
1,e006db26-a760-4bb5-8ba1-65b4a2b44c17,dicionario,2026-07-12T13:40:02.747503+00:00,2026-07-12T13:40:06.516643+00:00,SUCESSO,27,27,True,3.77,1174,None
2,e006db26-a760-4bb5-8ba1-65b4a2b44c17,meta_alfabetizacao_brasil,2026-07-12T13:40:06.516671+00:00,2026-07-12T13:40:10.147371+00:00,SUCESSO,3,3,True,3.63,270,None
3,e006db26-a760-4bb5-8ba1-65b4a2b44c17,meta_alfabetizacao_municipio,2026-07-12T13:40:10.147399+00:00,2026-07-12T13:40:13.031822+00:00,SUCESSO,10704,10704,True,2.88,1151232,None
4,e006db26-a760-4bb5-8ba1-65b4a2b44c17,meta_alfabetizacao_uf,2026-07-12T13:40:13.031850+00:00,2026-07-12T13:40:16.959713+00:00,SUCESSO,81,81,True,3.93,7374,None
5,e006db26-a760-4bb5-8ba1-65b4a2b44c17,municipio,2026-07-12T13:40:16.959735+00:00,2026-07-12T13:40:19.921346+00:00,SUCESSO,23995,23995,True,2.96,1832061,None
6,e006db26-a760-4bb5-8ba1-65b4a2b44c17,uf,2026-07-12T13:40:19.921374+00:00,2026-07-12T13:40:23.194925+00:00,SUCESSO,145,145,True,3.27,10330,None


4. QUALITY GATE E EXPORTAÇÃO DOS LOGS

In [6]:

from pathlib import Path


# ------------------------------------------------------------
# Finalização da execução
# ------------------------------------------------------------

finished_at = datetime.now(timezone.utc)

total_duration_seconds = round(
    (finished_at - started_at).total_seconds(),
    2
)

total_bytes_processed = int(
    batch_logs_df["bytes_processados"]
    .fillna(0)
    .sum()
)

total_mb_processed = round(
    total_bytes_processed / 1024**2,
    2
)

successful_tables = int(
    (batch_logs_df["status"] == "SUCESSO").sum()
)

failed_tables = int(
    (batch_logs_df["status"] == "ERRO").sum()
)

counts_ok = bool(
    batch_logs_df["contagem_confere"].all()
)

pipeline_success = (
    successful_tables == len(TABLES)
    and failed_tables == 0
    and counts_ok
)


# ------------------------------------------------------------
# Resumo executivo
# ------------------------------------------------------------

summary = {
    "execution_id": execution_id,
    "inicio_utc": started_at.isoformat(),
    "fim_utc": finished_at.isoformat(),
    "duracao_total_segundos": total_duration_seconds,
    "tabelas_esperadas": len(TABLES),
    "tabelas_com_sucesso": successful_tables,
    "tabelas_com_erro": failed_tables,
    "contagens_conferem": counts_ok,
    "mb_processados": total_mb_processed,
    "status_pipeline": (
        "SUCESSO"
        if pipeline_success
        else "FALHA"
    ),
}

summary_df = pd.DataFrame([summary])

display(summary_df)


# ------------------------------------------------------------
# Quality gate
# ------------------------------------------------------------

if not pipeline_success:
    raise RuntimeError(
        "Quality gate reprovado. "
        "Existem falhas ou divergências na carga batch."
    )

print("\nQUALITY GATE APROVADO")
print("Todas as entidades foram carregadas corretamente.")


# ------------------------------------------------------------
# Exportação dos logs
# ------------------------------------------------------------

logs_directory = Path("/content/logs")
logs_directory.mkdir(
    parents=True,
    exist_ok=True
)

safe_execution_id = execution_id.replace("-", "_")

detailed_log_path = (
    logs_directory
    / f"batch_detalhado_{safe_execution_id}.csv"
)

summary_log_path = (
    logs_directory
    / f"batch_resumo_{safe_execution_id}.csv"
)

json_log_path = (
    logs_directory
    / f"batch_detalhado_{safe_execution_id}.jsonl"
)

batch_logs_df.to_csv(
    detailed_log_path,
    index=False
)

summary_df.to_csv(
    summary_log_path,
    index=False
)

batch_logs_df.to_json(
    json_log_path,
    orient="records",
    lines=True,
    force_ascii=False,
)


print("\nLogs exportados:")
print(f"- {detailed_log_path}")
print(f"- {summary_log_path}")
print(f"- {json_log_path}")

,execution_id,inicio_utc,fim_utc,duracao_total_segundos,tabelas_esperadas,tabelas_com_sucesso,tabelas_com_erro,contagens_conferem,mb_processados,status_pipeline
0,e006db26-a760-4bb5-8ba1-65b4a2b44c17,2026-07-12T13:34:30.344736+00:00,2026-07-12T13:48:56.404605+00:00,866.06,7,7,0,True,258.97,SUCESSO



QUALITY GATE APROVADO
Todas as entidades foram carregadas corretamente.

Logs exportados:
- /content/logs/batch_detalhado_e006db26_a760_4bb5_8ba1_65b4a2b44c17.csv
- /content/logs/batch_resumo_e006db26_a760_4bb5_8ba1_65b4a2b44c17.csv
- /content/logs/batch_detalhado_e006db26_a760_4bb5_8ba1_65b4a2b44c17.jsonl
